In [0]:
%run ../01_Bronze/00_setup

In [0]:
# process new records into the Silver layer inventory table
from pyspark.sql import functions as F

def process_scd1_inventory(microBatchDF, batchId):
    target_table = "workspace.amazon_fullfilment_silver.inventory_silver"
    
    # 1. Deduplicate based on the COMPOSITE natural key
    # This ensures we only have one row per shelf/product combo per batch
    source_df = microBatchDF.dropDuplicates(["shelf_id", "product_id"])
    source_df.createOrReplaceTempView("source_inventory_view")

    # 2. Updated Merge Logic
    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_inventory_view AS source
        ON target.shelf_id = source.shelf_id 
           AND target.product_id = source.product_id
        
        -- We only update the 'Quantity' or 'Order_ID' if the composite key matches
        WHEN MATCHED AND (
            source.quantity <> target.quantity OR 
            source.order_id <> target.order_id
        ) THEN
            UPDATE SET 
                target.quantity = source.quantity,
                target.order_id = source.order_id,
                target.updated_timestamp = current_timestamp()
          
        WHEN NOT MATCHED THEN
          INSERT (
            inventory_sk, 
            shelf_id, 
            product_id, 
            quantity, 
            order_id, 
            updated_timestamp
          )
          VALUES (
            -- IMPORTANT: The SK must now be a hash of BOTH columns
            md5(concat(source.shelf_id, source.product_id)), 
            source.shelf_id, 
            source.product_id, 
            source.quantity, 
            source.order_id, 
            current_timestamp()
          )
    """)
    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.inventory")
    .filter("shelf_id IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd1_inventory)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_inventory_scd1")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())